[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begelb/latent_dynamics/blob/paper/notebooks/04_chafee_infante.ipynb)

In [ ]:
# Install CMGDB and the paper repository (the clone carries the saved model
# weights the notebooks recompute from). Running locally inside the repo,
# skip this cell.
!git clone -q --depth 1 --branch paper https://github.com/begelb/latent_dynamics.git
!pip install -q CMGDB
!pip install -q -e latent_dynamics
%cd latent_dynamics

# An editable install only becomes importable after a kernel restart, so
# import the package straight from the clone instead.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))

# Section 4.4 - Chafee-Infante PDE with bistability

## What this notebook shows

The **Chafee-Infante** semilinear parabolic PDE
$u_t = u_{xx} + \lambda(u - u^3)$ on $[0, \pi]$ with Dirichlet boundary
conditions (paper section 4.4). At $\lambda = 28$ the equation has eleven steady
states -- two stable, nine unstable. Discretizing in space yields a
**64-dimensional** system; we learn a two-dimensional latent model whose Morse
graph recovers the **bistability** (the two stable steady states $u_\pm$) as two
minimal nodes.

### How to run

Set `MODE` in the parameters cell, then Run All.

| `MODE` | what it does | typical cost |
|--------|--------------|--------------|
| `"quick"` | recompute the Morse graph of the *saved* model on the coarse `QUICK_SUBDIV` grid | seconds |
| `"morse"` | recompute the Morse graph of the *saved* model at your `SUBDIV` (paper value by default) | minutes |
| `"retrain"` | train a fresh model, then compute its Morse graph at your `SUBDIV` | minutes-hours (GPU recommended) |

**Coarse grids can merge nearby recurrent sets and change the Morse graph**,
so `quick` is a preview, not a paper-quality result. Nothing a notebook does
ever touches the preserved paper trees: recomputes land under
`output/notebooks/<experiment>/`.

> **On `retrain` for this example:** fresh retrains currently *overfit* and can
> fail the two-attractor ground truth, so the **released artifacts are the paper
> reference**. Retrain mode is provided for experimentation, not verification.
> Exact regions of attraction are off by default because they cost a second
> pass over the phase space; set `COMPUTE_ROA = True` in the
> parameters cell to compute them.

In [ ]:
# ===== PARAMETERS  (edit, then Run All) ====================================
MODE = "quick"             # "quick" | "morse" | "retrain"
SEED = None                # None -> the config's default seed
QUICK_SUBDIV = (10, 14, 20)  # MODE="quick": coarse preview grid, runs in seconds
SUBDIV = (10, 14, 28)      # MODE="morse"/"retrain": (subdiv_init, subdiv_min, subdiv_max)
                           # the paper value
COMPUTE_ROA = False        # exact regions of attraction from the map graph (extra pass)
# ===========================================================================

REPLAY_CONFIG  = "chafee_infante_replay"
RETRAIN_CONFIG = "chafee_infante"

## The system

The Chafee-Infante equation

$$u_t = u_{xx} + \alpha\,(u - u^3), \qquad x \in [0, \pi], \quad u(0) = u(\pi) = 0$$

discretized on `N` interior points and integrated for a fixed time `tau` per
step, which turns the PDE into a map on $\mathbb{R}^{N}$. At $\alpha = 28$ it
has eleven equilibria, two of them stable; the latent model has to find that
bistability from a 64-dimensional state.

In [ ]:
# ---- the system (paper values; edit and re-run to explore) ----------------
N_POINTS = 64        # spatial discretization
ALPHA = 28.0         # bifurcation parameter; 11 equilibria, 2 stable
TAU = 0.1            # integration time per step
AMPLITUDE = 2.0      # scale of the sampled initial conditions
DECAY = 0.5          # spectral decay of the sampled initial conditions
LATENT_DIMS = 2

from latentdynamics.config import load_config
from latentdynamics.systems import build_system

SYSTEM_PARAMS = {
    "N": N_POINTS, "alpha": ALPHA, "tau": TAU,
    "amplitude": AMPLITUDE, "decay": DECAY,
}
system = build_system("chafee_infante", SYSTEM_PARAMS)
print(f"ambient dimension {system.dim}, latent dimension {LATENT_DIMS}")

## The autoencoder and its latent map

An encoder, a decoder, and a latent map trained together so the latent map is
an $\epsilon$-approximate semiconjugacy to the full system on the data.

In [ ]:
# ---- the autoencoder (paper values) --------------------------------------
ENCODER_SHAPES = [64, 32]        # the encoder narrows 64 -> 2
LATENT_SHAPES = [32, 32]
DECODER_SHAPES = [32, 64]
HIDDEN_SHAPES = ENCODER_SHAPES   # compared against the paper below
LOSS_WEIGHTS = [1.0, 1.0, 0.0]   # (w1, w2, w3); the cycle term is off here
LEARNING_RATE = 3e-3
BATCH_SIZE = 30000               # full-batch: the dataset is small
EPOCHS = 4000
PATIENCE = 4000                  # equal to EPOCHS, i.e. early stopping disabled

print(f"{system.dim} -> {LATENT_DIMS} -> {system.dim}")
print(f"encoder {ENCODER_SHAPES}, latent map {LATENT_SHAPES}, decoder {DECODER_SHAPES}")
print(f"loss weights {LOSS_WEIGHTS}, Adam lr {LEARNING_RATE}, batch {BATCH_SIZE}")

## The data

Pairs $(x, f(x))$ along trajectories from sampled initial conditions: the model
only ever sees one-step transitions, never the map itself.

In [ ]:
# ---- the data (paper values) ---------------------------------------------
N_TRAIN = 1000        # training trajectories
N_VAL = 200           # validation trajectories
N_ITERATIONS = 30     # steps per trajectory

print(f"{N_TRAIN} train / {N_VAL} validation trajectories, {N_ITERATIONS} steps each")

# Everything above is fed to the pipeline as config overrides, so `retrain`
# below trains exactly the model described here.
OVERRIDES = {
    "system": {"params": SYSTEM_PARAMS},
    "arch": {
        "low_dims": LATENT_DIMS,
        "encoder": {"hidden_shapes": ENCODER_SHAPES},
        "latent_map": {"hidden_shapes": LATENT_SHAPES},
        "decoder": {"hidden_shapes": DECODER_SHAPES},
    },
    "training": {
        "loss_weights": LOSS_WEIGHTS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "patience": PATIENCE,
    },
    "data": {
        "n_samples_val": N_VAL,
        "n_iterations": N_ITERATIONS,
        
    },
}

# Flag anything that no longer matches the paper's configuration.
paper = load_config(RETRAIN_CONFIG)
drift = []
if SYSTEM_PARAMS != paper.system.params:
    drift.append(f"system {paper.system.params}")
if LATENT_DIMS != paper.arch.low_dims:
    drift.append(f"latent dims {paper.arch.low_dims}")
if HIDDEN_SHAPES != paper.arch.encoder.hidden_shapes:
    drift.append(f"encoder hidden {paper.arch.encoder.hidden_shapes}")
for name, value, reference in [
    ("loss weights", LOSS_WEIGHTS, paper.training.loss_weights),
    ("learning rate", LEARNING_RATE, paper.training.learning_rate),
    ("batch size", BATCH_SIZE, paper.training.batch_size),
    ("epochs", EPOCHS, paper.training.epochs),
    ("patience", PATIENCE, paper.training.patience),
    ("validation trajectories", N_VAL, paper.data.n_samples_val),
    ("iterations", N_ITERATIONS, paper.data.n_iterations),
]:
    if value != reference:
        drift.append(f"{name} {reference}")
print("\nmatches the paper's configuration" if not drift
      else "\ndiffers from the paper, which uses: " + "; ".join(drift))

## Training

`quick` and `morse` load the paper's trained weights. `retrain` runs the data,
scaling, training and diagnostic stages; CMGDB then runs below on the
fresh model, exactly as in the other modes.

In [ ]:
from latentdynamics.replay import load_experiment, retrain

if MODE in ("quick", "morse"):
    exp = load_experiment(REPLAY_CONFIG, seed=SEED)
elif MODE == "retrain":
    # Training only; CMGDB runs below on the fresh model, exactly as in the
    # other modes.
    exp = retrain(
        RETRAIN_CONFIG,
        seed=SEED,
        overrides=OVERRIDES,
        stages=("data", "scale", "train", "diagnose"),
    )
else:
    raise ValueError(f"unknown MODE {MODE!r}")
exp

## Training curves

Total loss and its terms, per epoch. In `quick` and `morse` mode these are the
saved curves of the run the paper reports; in `retrain` mode they are the run
that just finished. Training stops early when validation loss stalls, so the
curves usually end well before the configured epoch budget.

In [ ]:
import json

import matplotlib.pyplot as plt

history_path = exp.seed_dir / "logs" / "history.json"
if not history_path.exists():
    print(f"no training history at {history_path}")
else:
    history = json.loads(history_path.read_text())
    terms = [k for k in history["train"] if k != "loss_total"]

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
    left.semilogy(history["train"]["loss_total"], label="train")
    left.semilogy(history["val"]["loss_total"], label="validation")
    left.set(xlabel="epoch", ylabel="total loss")
    left.legend()

    for term in terms:
        right.semilogy(history["val"][term], label=term)
    right.set(xlabel="epoch", ylabel="validation loss by term")
    right.legend(fontsize="small")

    fig.tight_layout()
    plt.show()
    print(f"{len(history['train']['loss_total'])} epochs")

## Morse graph

CMGDB subdivides the latent rectangle into boxes and builds the directed graph
on them induced by the latent map: box `B` points at every box meeting an
enclosure of `g(B)`. The **Morse sets** are that graph's strongly connected
components, and the **Morse graph** is its condensation, with a Conley index on
each node. `ComputeConleyMorseGraph` returns both -- the map graph is not an
extra step, it *is* the computation.

The grid comes from `QUICK_SUBDIV` in `quick` mode and from `SUBDIV`
otherwise, chosen here rather than inherited from the training config.

The box map evaluates the network on the corner lattice in batches -- one call
per batch instead of one per box. CMGDB (>= 1.5.0) caches the transition graph in one block with
automatic edge reservation, so no environment tuning is needed; deep grids
are limited by memory alone.

In [ ]:
# Recompute the Morse graph of the loaded (or freshly trained) model. The
# artifacts land in the notebook playground, never the preserved paper trees,
# and the returned run keeps the live CMGDB objects for the plots below.
subdiv = QUICK_SUBDIV if MODE == "quick" else SUBDIV
run = exp.recompute_morse(subdiv=subdiv, cmgdb_overrides={"compute_roa": COMPUTE_ROA})
MG_DIR = run.morse_dir
print(f"artifacts -> {MG_DIR}")

### What came out

Each Morse set with its Conley index, how many boxes it occupies, and what it
flows into. A node with no outgoing edges is minimal: an attractor.

In [ ]:
import numpy as np

from latentdynamics.analysis import MorseGraph

graph = MorseGraph.from_dot(MG_DIR / "morse_graph")
boxes = np.atleast_2d(np.loadtxt(MG_DIR / "morse_sets", delimiter=","))
counts = dict(zip(*np.unique(boxes[:, -1].astype(int), return_counts=True)))

print(f"{len(graph.nodes)} Morse sets, {len(graph.minimal)} minimal")
for node in graph.nodes:
    index = graph.labels.get(node, "?").split(":", 1)[-1].strip()
    edges = sorted(graph.edges.get(node, []))
    flow = "minimal" if not edges else "-> " + ", ".join(str(e) for e in edges)
    print(f"  {node}: {index:<16} {counts.get(node, 0):>8d} boxes   {flow}")

## Figures

Rendered from the DOT and CSV above with the paper's palette and axis labels,
so a recomputed run and the paper figure come out of the same code. `BOX_SCALE`
only affects drawing: it inflates Morse sets too small to see.

In [ ]:
import CMGDB

CMGDB.PlotMorseGraph(run.cmgdb_morse_graph)

In [ ]:
CMGDB.PlotMorseSets(run.cmgdb_morse_graph, xlabel="$z_1$", ylabel="$z_2$");

## Regions of attraction

This is what the map graph is for. Each of its vertices is a box; following the
edges tells you which minimal Morse sets a box can reach, and a box that can
reach exactly one of them lies in that attractor's basin. Boxes reaching
several are the boundary between basins.

Computing them costs a second pass over the phase space on top of the Morse
graph, so they are opt-in via `COMPUTE_ROA` in the parameters cell.

In [ ]:
# Exact regions of attraction, computed from the map graph when COMPUTE_ROA
# was set in the parameters cell.
from IPython.display import Image, display

roa_pngs = sorted(MG_DIR.glob("regions_of_attraction*.png"))
if roa_pngs:
    for png in roa_pngs:
        display(Image(filename=str(png)))
else:
    print("no regions-of-attraction figure for this run; "
          "set COMPUTE_ROA = True in the parameters cell and rerun")

## Run provenance

In [ ]:
run.diagnostics()